In [3]:
import mediapipe as mp
import cv2
import numpy as np
import time
import threading

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

model_path = "face_landmarker.task"

latest_result = None
result_lock = threading.Lock()

def store_result(result, output_image, timestamp_ms):
    global latest_result
    with result_lock:
        latest_result = result

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.LIVE_STREAM,
    result_callback=store_result
)


In [ ]:
from scipy.signal import butter, filtfilt

def chrom(rgb_signal):
    r = rgb_signal[:, 0]
    g = rgb_signal[:, 1]
    b = rgb_signal[:, 2]

    #normalize
    total = r + g + b + 1e-6 
    r_n = r / total
    g_n = g / total
    b_n = b / total

    Xs = 3 * r_n - 2 * g_n
    Ys = 1.5 * r_n + g_n - 1.5 * b_n

    std_Xs = np.std(Xs) + 1e-6
    std_Ys = np.std(Ys) + 1e-6
    pulse = Xs - (std_Xs / std_Ys) * Ys

    return pulse

def bandpass(signal, fps, low=0.7, high=4.0):
    nyq = fps / 2.0
    low_n = low / nyq
    high_n = high / nyq
    b, a = butter(4, [low_n, high_n], btype='band')
    return filtfilt(b, a, signal)



In [4]:
cap = cv2.VideoCapture(0)
start_time = time.time()

time.sleep(1)

with FaceLandmarker.create_from_options(options) as landmarker:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        timestamp_ms = int((time.time() - start_time) * 1000)
        landmarker.detect_async(mp_image, timestamp_ms)

        with result_lock:
            current_result = latest_result

        if current_result and current_result.face_landmarks:
            landmarks = current_result.face_landmarks[0]
            h, w, _ = frame.shape

            for lm in landmarks:
                x, y = int(lm.x * w), int(lm.y * h)
                cv2.circle(frame, (x, y), 1, (0, 255, 0), -1)

    
        regions = {
            "forehead": [109, 10, 338, 336, 9, 107],
            "left_cheek": [116, 111, 117, 118, 119, 120, 100, 142, 36, 205, 123],
            "right_cheek": [371, 329, 349, 348, 347, 346, 340, 345, 352, 425, 266]
        }

        mask = np.zeros(frame.shape[:2], dtype=np.uint8)

        for region_name, ids in regions.items():
            points = np.array([
                (int(landmarks[i].x * w), int(landmarks[i].y * h))
                for i in ids
            ])
            cv2.polylines(frame, [points], isClosed=True, color=(0, 0, 255), thickness=2)
            cv2.fillPoly(mask, [points], 255)

        mean_bgr = cv2.mean(frame, mask=mask)[:3]
        print(f"B: {mean_bgr[0]:.1f}, G: {mean_bgr[1]:.1f}, R: {mean_bgr[2]:.1f}")
        
        cv2.imshow("rPPG", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

W0000 00:00:1781248175.923174 17155950 face_landmarker_graph.cc:180] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
I0000 00:00:1781248175.931285 17155950 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3 Pro
W0000 00:00:1781248175.932795 17155952 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781248175.944872 17155958 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


B: 47.0, G: 54.3, R: 90.7
B: 58.4, G: 69.5, R: 121.0
B: 57.3, G: 68.3, R: 118.4
B: 57.0, G: 67.9, R: 117.8
B: 56.8, G: 67.8, R: 117.7
B: 56.6, G: 67.7, R: 117.7
B: 56.6, G: 67.9, R: 117.9
B: 56.4, G: 67.7, R: 117.6
B: 56.8, G: 67.7, R: 118.0
B: 56.7, G: 67.7, R: 118.1
B: 56.6, G: 67.6, R: 118.0
B: 56.8, G: 67.9, R: 118.3
B: 56.8, G: 67.8, R: 118.2
B: 56.9, G: 67.7, R: 118.0
B: 57.0, G: 67.8, R: 118.0
B: 56.9, G: 67.8, R: 117.7
B: 56.8, G: 67.8, R: 117.6
B: 56.7, G: 67.8, R: 117.5
B: 56.7, G: 67.8, R: 117.4
B: 57.0, G: 67.9, R: 117.8
B: 56.8, G: 67.8, R: 118.1
B: 56.8, G: 67.6, R: 117.9
B: 56.8, G: 67.5, R: 118.1
B: 57.1, G: 67.5, R: 118.2
B: 57.1, G: 67.4, R: 117.8
B: 56.9, G: 67.6, R: 117.8
B: 56.9, G: 67.4, R: 117.7
B: 56.7, G: 67.3, R: 117.6
B: 56.8, G: 67.3, R: 117.5
B: 56.8, G: 67.3, R: 117.4
B: 56.5, G: 67.4, R: 117.6
B: 56.4, G: 67.6, R: 117.7
B: 56.4, G: 67.5, R: 117.8
B: 56.8, G: 67.5, R: 117.5
B: 56.5, G: 67.6, R: 117.6
B: 56.8, G: 67.8, R: 117.8
B: 56.4, G: 67.9, R: 117.8
B:

KeyboardInterrupt: 